# Your First Vector RAG Application: Cat Health Assistant

In this notebook, we will build a dense vector retrieval application using **LangChain v1**, **OpenAI embeddings**, and **Qdrant** as an in-memory vector database.

The goal is to understand the core RAG loop:

1. Load a cat health guideline PDF
2. Split it into smaller chunks
3. Embed those chunks
4. Store the embeddings in Qdrant
5. Retrieve relevant chunks for a question
6. Generate an answer grounded in the retrieved context

> Note: This notebook expects Python 3.12 and uses uv for dependency management.

> Note: This is a vector RAG lesson, not a veterinary care tool. The assistant should answer from the PDF and point users to a veterinarian for diagnosis, treatment, medication, or urgent care decisions.

## Table of Contents

- Task 1: Environment Setup
- Task 2: Embedding Similarity Primer
- Task 3: Documents - Loading the Cat Health Guideline PDF
- Task 4: Chunking the Documents
- Task 5: Embeddings and Qdrant
- Task 6: Retrieval with Scores
- Task 7: Retrieval Augmented Generation
- Activity: Tune Retrieval Quality

## Task 1: Environment Setup

From the `01_Dense_Vector_Retrieval` folder, install dependencies with uv:

```bash
uv sync
```

Then open this notebook in Cursor or VS Code and select the Python/Jupyter environment created by uv.

### Imports

LangChain v1 separates integrations into partner packages. We will use:

- `langchain_community` for PDF loading
- `langchain_text_splitters` for chunking
- `langchain_openai` for chat and embedding models
- `langchain_qdrant` for the Qdrant vector store

In [1]:
from pathlib import Path
from math import sqrt
from getpass import getpass
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

/tmp/ipykernel_10529/4147450701.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### OpenAI API Key

The chat model and embedding model both use OpenAI. If `OPENAI_API_KEY` is not already set in your environment, this cell will ask for it securely.

In [2]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

## Task 2: Embedding Similarity Primer

Before we load a full PDF, let's make dense vector retrieval less mysterious.

An embedding model turns text into a list of numbers. Texts with related meaning should land closer together in that vector space.

A common way to score closeness is **cosine similarity**:

```text
cosine_similarity(a, b) = dot_product(a, b) / (length(a) * length(b))
```

The intuition: if two vectors point in a similar direction, their cosine similarity is higher. Vector databases like Qdrant use this same idea, but at a much larger scale.

In [3]:
embedding_model = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=embedding_model)

example_texts = [
    "king",
    "queen",
    "banana",
    "cat",
    "veterinarian",
    "cat health guidelines",
]

example_vectors = dict(zip(example_texts, embeddings.embed_documents(example_texts)))


def cosine_similarity(vector_a: list[float], vector_b: list[float]) -> float:
    dot_product = sum(a * b for a, b in zip(vector_a, vector_b))
    length_a = sqrt(sum(a * a for a in vector_a))
    length_b = sqrt(sum(b * b for b in vector_b))
    return dot_product / (length_a * length_b)


comparison_pairs = [
    ("king", "queen"),
    ("king", "banana"),
    ("cat", "veterinarian"),
    ("cat", "cat health guidelines"),
]



for left, right in comparison_pairs:
    score = cosine_similarity(example_vectors[left], example_vectors[right])
    print(f"{left:>22} <> {right:<22} score={score:.3f}")

                  king <> queen                  score=0.591
                  king <> banana                 score=0.310
                   cat <> veterinarian           score=0.356
                   cat <> cat health guidelines  score=0.496


A few important notes:

- The score is useful for ranking, not as an absolute truth about meaning.
- Different embedding models can produce different scores.
- In RAG, we embed each document chunk once, then embed the user's query and search for the nearest chunk vectors.

That is the retrieval part of RAG.

#### ❓Question #1

Why is cosine similarity useful for dense vector retrieval?

##### ✅ Answer:

Cosine similarity is useful for dense vector retrieval because we know that dense vectors with similar angles have similar textual meaning regardless of their length, and if we run cosine similarity between a prompt (in embedded form) and parts [chunks] of a reference document (also in embedded form), we can find this way only the parts of the document which should have the most similar meaning, while normalizing the short question with the much longer parts of the document.
The cosine gives a clean and bounded score (between -1 and 1), which means its easy to rank the retrieved vectors. This is all provided that the embedding was done well. Also, when vectors are normalized to length 1 before the operation, the Cosine similarity becomes very efficient computationally, (simple dot product, multiply and add, without square roots).

## Task 3: Documents

LangChain represents loaded text as `Document` objects. A `Document` has:

- `page_content`: the text
- `metadata`: information such as source file and page number

We will load one `Document` per PDF page, then split those pages into smaller chunks.

### Course PDF

This notebook uses the bundled cat health guideline PDF at:

```text
01_Dense_Vector_Retrieval/data/cat_health_guidelines.pdf
```

The next cell checks that the course material is present before we start loading pages.

In [4]:
pdf_path = Path("data/cat_health_guidelines.pdf")

if not pdf_path.exists():
    raise FileNotFoundError(
        f"Expected the cat health guideline PDF at: {pdf_path.resolve()}\n"
        "The bundled course PDF is missing from this copy of the materials."
    )

### Load the PDF

`PyPDFLoader` extracts text from text-based PDFs. If the PDF is scanned images, this loader may return little or no text, and OCR would be needed.

In [5]:
loader = PyPDFLoader(str(pdf_path))
pages = loader.load()

for page in pages:
    page.metadata["source"] = pdf_path.name
    page.metadata["document_type"] = "cat_health_guideline"

pages = [page for page in pages if page.page_content.strip()]

if not pages:
    raise ValueError(
        "The PDF loaded, but no extractable text was found. "
        "This usually means the PDF is scanned and needs OCR first."
    )

print(f"Loaded {len(pages)} text-containing PDF pages.")

Loaded 22 text-containing PDF pages.


In [ ]:
print(pages[0].page_content[:750])
print("\nMetadata:", pages[0].metadata)

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and theJournal of the American Animal Hospital
Association(volume 57, issue 2, pages 51–72, DOI: 10.5326/JAAHA-MS-7189). A

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'so

#### ❓Question #2

Why is metadata important for a RAG application?

##### ✅ Answer:

Metadata is important for rag applications because embedded chunks for documents don't hold information like recency (the date of the document), version numbers, document page numbers, etc. This means, that once you augment your prompt, without metadata you can't cite sources or filter out data that might have similar meaning to the query, but is outdated or irrelevant to the topic. Another important aspect is permissions. Some data shouldn't be available to users depending on their clearance. By attaching metadata to embedded chunks, you can have control over what users are allowed to retrieve. If RAG uses a metadata filter to narrow down to the chunks that are current, on-topic, and permitted, and then runs cosine similarity, the most meaningful matches within that filtered set can be found.

## Task 4: Chunking the Documents

A full PDF page can be too large or too mixed-topic for high-quality retrieval. We split pages into overlapping chunks so each chunk has enough local context but is still focused.

Here we will start with chunks of 1,000 characters and 200 characters of overlap. The chunk size controls how much text each vector represents; the overlap keeps nearby context from being lost at chunk boundaries.

`RecursiveCharacterTextSplitter` tries to split on natural boundaries first, such as paragraphs and line breaks, before falling back to smaller separators.

In [6]:
chunk_size = 1000
chunk_overlap = 200

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    add_start_index=True,
)

splits = text_splitter.split_documents(pages)

print(f"Split {len(pages)} pages into {len(splits)} chunks.")
print(f"Chunk size: {chunk_size} characters")
print(f"Chunk overlap: {chunk_overlap} characters")

Split 22 pages into 135 chunks.
Chunk size: 1000 characters
Chunk overlap: 200 characters


In [27]:
sample_chunk = splits[0]
print(sample_chunk.page_content[:750])
print("\nMetadata:", sample_chunk.metadata)

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and theJournal of the American Animal Hospital
Association(volume 57, issue 2, pages 51–72, DOI: 10.5326/JAAHA-MS-7189). A

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'so

Use the [Chunk Visualizer](https://chunkviz.up.railway.app/) to experiment with different chunk sizes and overlaps and see how the text boundaries change.

#### ❓Question #3

What tradeoff do we make when choosing chunk size and chunk overlap?

##### ✅ Answer:

When choosing chunk size, a large chunk means that even though it will more likely hold relevant contextual information to the query, it will also hold much irrelevant information, and when embedding it, it might average out in vector similarity, and get ranked low. Also, it will inflate the context window. On the other hand, a small chunk might be too small to have more than a single narrow idea, but it could potentially point to a similar direction to the query, resulting in a a highly ranked vector, but not enough context to stand on its own, rendering it useless.  When choosing chunk overlap, if the overlap is too small, ideas might get cut-off at the  beginning or end of chunks, losing valuable context. but if the overlap is too big, then it introduces redundancy. The same text will be introduced multiple times, costing with inefficient context window.



## Task 5: Embeddings and Qdrant

Now we apply the same embedding idea to every chunk from the PDF. Qdrant stores those vectors and lets us search for chunks that are close to a query in embedding space.

We already created an OpenAI embedding model in the primer above. The Qdrant collection name is just a label for the set of vectors we are creating.

For this notebook, Qdrant runs in memory with `location=":memory:"`. That means no Docker, no Qdrant Cloud account, and no persistence after the notebook kernel stops.

In [7]:
collection_name = "cat_health_guidelines"

vector_store = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=embeddings,
    location=":memory:",
    collection_name=collection_name,
    force_recreate=True,
)

print(f"Embedded chunks with: {embedding_model}")
print(f"Built in-memory Qdrant collection: {collection_name}")

Embedded chunks with: text-embedding-3-small
Built in-memory Qdrant collection: cat_health_guidelines


## Task 6: Retrieval with Scores

Before we generate answers, we should inspect retrieval directly. If retrieval returns poor context, the final answer will usually be poor too.

The value `k` controls how many chunks the retriever returns. A larger `k` gives the model more context, but it can also add noise. We will start with `k = 4` and tune it later.

Qdrant can return both the matching `Document` and a similarity score. This is the same ranking idea we saw with `king`, `queen`, and `cat`, now applied to PDF chunks.

In [8]:
def display_retrieval_results(query: str, k: int) -> list[tuple]:
    """Retrieve chunks and print a compact view of the results."""
    results = vector_store.similarity_search_with_score(query, k=k)

    for index, (doc, score) in enumerate(results, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        start_index = doc.metadata.get("start_index", "unknown")
        preview = doc.page_content[:350].replace("\n", " ")

        print(f"Source {index} | score={score:.3f} | page={page_display} | start_index={start_index}")
        print(preview)
        print("-" * 80)

    return results

In [ ]:
retrieval_k = 4
retrieval_query = "What should I know about feeding a healthy adult cat?"
retrieved_results = display_retrieval_results(retrieval_query, k=retrieval_k)

Source 1 | score=0.380 | page=10 | start_index=1637
Young Adult Cats Y oung adult cats do not require as frequent routine medical care as kittens, so it is integral to educate the client about why regular healthcare examinations remain so important. Routine examinations can help identify behavioral changes or medical concerns that may affect a cat ’s health long before they become signi ﬁcant, painf
--------------------------------------------------------------------------------
Source 2 | score=0.379 | page=20 | start_index=2443
silvestris catus) kept in the home. Appl Anim Behav Sci 2005;93:97–109. 62. Clancy EA, Moore AS, Bertone ER. Evaluation of cat and owner characteristics and their relationships to outdoor access of owned cats. J Am V et Med Assoc 2003;222:1541–5. 63. Neville PF. An ethical viewpoint: the role of veterinarians and behaviourists in ensuring good husb
--------------------------------------------------------------------------------
Source 3 | score=0.367 | page=19

#### ❓Question #4

What does a similarity score help you understand, and what does it not prove by itself?

##### ✅ Answer:

A similarity score helps me understand whether two texts have similar textual meaning. A high score means their embedded vector representations are angularly close. But it doesn't prove the following:

1. It doesn't check if a retrieved chunk has true, false, accurate or recent information.
2. It can't check if retrieved chunks with high scores have contradictory information.
3. A similarity score doesn't take into account model biases or blind spots. Also, it depends on the model what is considered a high score.
4. A similarity score doesn't take metadata into account, which we already know is crucial for relevance of retrieved chunks.

## Task 7: Retrieval Augmented Generation

Now we combine retrieval with generation. We will use a two-step RAG pattern:

1. Retrieve relevant chunks from Qdrant
2. Put those chunks into the prompt and ask the model to answer from the context

This is intentionally simpler than an agent. We always retrieve before answering, which makes the vector retrieval mechanics easy to inspect.

For generation, we will use `gpt-5.4-mini`.

In [9]:
chat_model = "gpt-5.4-mini"
llm = ChatOpenAI(model=chat_model)

RAG_SYSTEM_PROMPT = """You are a cat health guideline assistant in a vector RAG lesson.

Use only the provided context to answer the user's question.
If the context does not contain enough information, say: "I don't have enough information in the provided cat health guideline PDF to answer that."

Cite the retrieved sources inline using labels like [Source 1] or [Source 2].
Do not diagnose, prescribe medication, or replace a veterinarian.
For diagnosis, treatment decisions, medication questions, or urgent symptoms, recommend contacting a veterinarian.
Keep the answer concise and practical."""

RAG_USER_PROMPT = """Context:
{context}

Question: {question}

Answer from the context above."""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", RAG_USER_PROMPT),
    ]
)

rag_chain = rag_prompt | llm | StrOutputParser()

In [10]:
def format_context(scored_docs: list[tuple]) -> str:
    """Convert retrieved documents into a source-labeled context string."""
    formatted_chunks = []

    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        source = doc.metadata.get("source", "unknown source")

        formatted_chunks.append(
            f"[Source {index}] {source}, page {page_display}, score {score:.3f}\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(formatted_chunks)


def answer_question(question: str, k: int) -> dict:
    """Run retrieve-then-generate and return the answer plus source metadata."""
    scored_docs = vector_store.similarity_search_with_score(question, k=k)
    context = format_context(scored_docs)
    answer = rag_chain.invoke({"context": context, "question": question})

    sources = []
    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        sources.append(
            {
                "source_label": f"Source {index}",
                "file": doc.metadata.get("source"),
                "page": page + 1 if isinstance(page, int) else None,
                "start_index": doc.metadata.get("start_index"),
                "score": score,
            }
        )

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "context": scored_docs,
    }

Before calling the model, inspect the formatted context. This is the exact text that will be inserted into the RAG prompt.

In [39]:
example_context = format_context(retrieved_results[:4])
print(example_context[:4000])

[Source 1] cat_health_guidelines.pdf, page 14, score 0.663
good starting point is to calculate the adult feline patient ’s resting
energy requirements (RER) according to the following calculation:
RER (kcal per day) ¼ 30 × (body weight in kg) 1 70. Daily energy
requirements (DER) are determined based on multiplying by a
needs factor, which in the case of young, healthy adults is 1. Food
intake can be determined by comparing DER with the caloric
density of the patient ’s foods.
85,98–100
Prescription diets are indicated for obesity treatment. These
weight loss diets are formulated to provide adequate vitamins and
minerals with reduced caloric content. It is important to inform
owners of overweight cats that simply feeding less of a maintenance
diet in order to reduce caloric intake may result in vitamin and
mineral de ﬁciencies.
Mature Adult and Senior Cats
Mature adult and senior cats have changing dietary needs, and it is
extremely important to provide guidance on daily feeding amount

In [18]:
answer_k = 4

result = answer_question(
    "What are signs that my cat may need veterinary attention?",
    k=answer_k,
)

print(result["answer"])
print("\nSources:")
for source in result["sources"]:
    print(source)

Signs that may warrant veterinary attention include changes in grooming, especially increased or reduced grooming, which can be linked to skin disease, illness, bladder pain, joint pain, or reduced mobility [Source 1][Source 2]. Other concerning changes include vomiting, vomiting hairballs, diarrhea, changes in appetite, increased drinking or urination, increased nocturnal activity or vocalization, and changes in normal habits or activity [Source 3]. Signs of pain, anxiety, stress, or fear can include cowering, crouching, hiding, freezing, tense body posture, flat or rotated ears, dilated pupils, forward whiskers, and hissing, yowling, growling, or screaming [Source 4]. If you notice these signs, contact a veterinarian for evaluation.

Sources:
{'source_label': 'Source 1', 'file': 'cat_health_guidelines.pdf', 'page': 8, 'start_index': 0, 'score': 0.5961885722878416}
{'source_label': 'Source 2', 'file': 'cat_health_guidelines.pdf', 'page': 8, 'start_index': 767, 'score': 0.5683504240474

### Vibe Check Queries

Run a few questions that should be answerable from a cat health guideline PDF. Then run one question that may not be answerable and confirm the assistant says it does not have enough information.

In [ ]:
vibe_check_questions = [
    "What preventive care is recommended for cats?",
    "What symptoms should make me call a veterinarian?",
    "What should I know about feeding a healthy adult cat?",
    "Can my cat help me file my taxes?",
]

for question in vibe_check_questions:
    print("Question:", question)
    print(answer_question(question, k=answer_k)["answer"])
    print("=" * 100)

Question: What preventive care is recommended for cats?
The guidelines recommend **at least annual veterinary examinations for all cats**, with more frequent visits based on the cat’s individual needs; **senior cats should be seen at least every 6 months** [Source 4]. Preventive care also includes **individualized risk assessment and preventive healthcare strategies**, with **routine use of broad-spectrum parasite control** beneficial for most pet cats [Source 1]. In addition, the guidelines note that topics like **sterilization, claw care, identification/microchipping, and disaster preparedness** may be covered during preventive care visits [Source 4].
Question: What symptoms should make me call a veterinarian?
According to the guideline, you should contact a veterinarian if your cat has changes in appetite, increased drinking or urination, vomiting, vomiting hairballs, diarrhea, weight changes, or changes in normal habits/activity such as increased nocturnal activity or vocalization 

#### ❓Question #5

For the vibe check queries above, did the retrieved context seem relevant before generation? Why or why not?

##### ✅ Answer:

For query 1, "What preventive care is recommended for cats?":
Sources 1 and 4 are relevant, as they give concrete examples of preventive care routines.
Source 2 isn't relevant because it discusses preventive care in general, but doesn't provide actual ways to execute preventive care. Source 3 isn't relevant because its a chunk containing references of the original document (a reference to references) without actual context. 

For query 2, "What symptoms should make me call a veterinarian?":
Source 1 is relevant,as it mentions symptoms of ilnesses that a veterinarian should instruct a cat owner not to ignore. 
Source 2 doesn't seem to be relevant, as it doen't answer the query but discusses why cat owner are sometimes reluctant to take senior cats to vets. (The generation decided that the mentioned symptoms "reduced jumping or climbing" are a reason to call a vet but the source doesn't state that.)
Source 3 is relevant as its actually the next sequential chunk after source 1, and it directly explains why the symptoms mentioned in source 1 are serious.
Source 4 seems to be relevant because it mentions "Changes in demeanor, activity level, and behavior are additionally key to note
and trend over time.", (The generation for some reason decided not to use it as a source, maybe because some of these symptoms were already mentioned in source 1)

For query 3, "What should I know about feeding a healthy adult cat?":
Source 1 seems relevant as it discusses in how to measure the caloric needs of cats. 
Source 2 is relevant because it instructs what nutrition any cat should consume, including healthy adult cats.
Source 3 seems relevant as it discusses young energy requirements and feeding amounts for young adult cats, which should fall to the category of "healthy adult cats".
Source 4 shouldn't be relevant as it discusses feeding sick cats (patients) or kittens, despite that the generated answer decided to use it as a source 

For query 4, "Can my cat help me file my taxes?":
As expected, none of the 4 chunks had could possibly have any relevant context. The llm also correctly reasoned that neither of the chunks answered this question. 

## 🏗️ Activity: Tune Retrieval Quality

Improve retrieval quality by changing one or more of these values:

- The chunk size
- The chunk overlap
- The retrieval `k`
- The wording of the retrieval query

Suggested workflow:

1. Pick one test question.
2. Inspect the retrieved chunks and scores.
3. Change one retrieval setting.
4. Rebuild the splitter and vector store.
5. Compare whether the retrieved chunks became more relevant.

When you are done, write down what changed and whether the final answer improved.

In [ ]:
# Activity workspace
# Try retrieval tuning here.

### YOUR CODE HERE

In [13]:
# --- Retrieve: cheap. Change k/query and re-run. Duplicate this cell to compare runs side by side. ---
exp_k = 4
exp_query = "What preventive care is recommended for cats?"

print(f"query: {exp_query} | k={exp_k}\n")
exp_results = display_retrieval_results(exp_query, k=exp_k)   # or: print(answer_question(exp_query, exp_k)["answer"])

exp_full_context = format_context(exp_results[:4])
print(exp_full_context[:4000])

query: What preventive care is recommended for cats? | k=4

Source 1 | score=0.657 | page=15 | start_index=1549
17,120,121 This approach will eliminate existing infections, as well as decrease the risk of further infestation and subsequent associated clinical problems. Canine and feline housemates may be at risk of transmission of infectious parasites including roundworm and ﬂeas and therefore should be treated in synchronicity with newly acquired kittens or
--------------------------------------------------------------------------------
Source 2 | score=0.648 | page=2 | start_index=821
alized risk assessment, preventive healthcare strategies, and treat- ment pathways that evolve as the cat matures. An evidence-guided framework for managing a cat ’s healthcare throughout its lifetime has never been more important in feline practice than it is now. Cats are the most popular pet in the United States. 1 A great anomaly in feline prac
-------------------------------------------------------

In [18]:
# --- Answer: Test of the llm's ability to answer the question based on the retrieved chunks ---
answer_k = 4

result = answer_question(
    "What preventive care is recommended for cats?",
    k=answer_k,
)

print(result["answer"])
print("\nSources:")
for source in result["sources"]:
    print(source)

The guideline recommends at least annual veterinary examinations for all cats, with more frequent visits based on individual needs; senior cats should be seen at least every 6 months [Source 4]. It also highlights preventive care topics such as sterilization, claw care, identification/microchipping, and disaster preparedness [Source 4]. In addition, routine broad-spectrum parasite prevention is likely beneficial for most pet cats, and cats with higher-risk lifestyles or exposure may need tailored parasite control [Source 1].

Sources:
{'source_label': 'Source 1', 'file': 'cat_health_guidelines.pdf', 'page': 15, 'start_index': 1549, 'score': 0.657266156381088}
{'source_label': 'Source 2', 'file': 'cat_health_guidelines.pdf', 'page': 2, 'start_index': 821, 'score': 0.6480087738852278}
{'source_label': 'Source 3', 'file': 'cat_health_guidelines.pdf', 'page': 19, 'start_index': 1659, 'score': 0.6267747067750677}
{'source_label': 'Source 4', 'file': 'cat_health_guidelines.pdf', 'page': 6, '

In [19]:
# --- Rebuild: run ONLY when you change chunk_size or chunk_overlap (re-embeds all chunks) ---
exp_chunk_size = 1500
exp_chunk_overlap = 200

exp_splitter = RecursiveCharacterTextSplitter(
    chunk_size=exp_chunk_size,
    chunk_overlap=exp_chunk_overlap,
    add_start_index=True,
)
exp_splits = exp_splitter.split_documents(pages)
vector_store = QdrantVectorStore.from_documents(
    documents=exp_splits,
    embedding=embeddings,
    location=":memory:",
    collection_name="cat_health_guidelines",
    force_recreate=True,
)
print(f"Rebuilt: chunks={len(exp_splits)} | size={exp_chunk_size} overlap={exp_chunk_overlap}")

Rebuilt: chunks=90 | size=1500 overlap=200


In [22]:
# --- Retrieve: cheap. Change k/query and re-run. Duplicate this cell to compare runs side by side. ---
exp_k = 4
exp_query = "What preventive care is recommended for cats?"

print(f"query: {exp_query} | k={exp_k}\n")
exp_results = display_retrieval_results(exp_query, k=exp_k)   # or: print(answer_question(exp_query, exp_k)["answer"])

exp_full_context = format_context(exp_results[:4])
print(exp_full_context[:6000])

query: What preventive care is recommended for cats? | k=4

Source 1 | score=0.654 | page=15 | start_index=1288
117–119 Parasite Control For kittens and newly adopted cats with an unknown history of medical care, it is prudent to administer prophylactic treatment for parasites with broad-spectrum products ef ﬁcacious against heart- worms, intestinal parasites, and ﬂeas. 17,120,121 This approach will eliminate existing infections, as well as decrease the risk
--------------------------------------------------------------------------------
Source 2 | score=0.636 | page=15 | start_index=2615
appropriate, may diagnose speci ﬁc infections and guide therapy; however, negative testing does not rule out infection. Ectoparasite prevention will lower the risk of cutaneous and systemic diseases. 120 As tick populations increase in number and expand geographically, the prevention of tick infestations in cats is becoming increasingly important. 
-----------------------------------------------------

In [21]:
# --- Answer: Test of the llm's ability to answer the question based on the retrieved chunks ---
answer_k = 4

result = answer_question(
    "What preventive care is recommended for cats?",
    k=answer_k,
)

print(result["answer"])
print("\nSources:")
for source in result["sources"]:
    print(source)

Preventive care for cats in the provided guideline includes:

- **Parasite control:** routine broad-spectrum parasite prevention is recommended for most pet cats, especially kittens and newly adopted cats with unknown medical history; this helps protect against heartworms, intestinal parasites, fleas, and ticks [Source 1][Source 2].
- **Treat housemates when needed:** canine and feline housemates may also need parasite treatment at the same time to reduce transmission risk [Source 1].
- **Environmental precautions:** limiting access to gardens and children’s sand areas can reduce contamination with parasites and zoonotic agents [Source 1].
- **Vaccination:** cats should receive **individualized vaccination protocols** based on lifestyle, life stage, origin, and risk factors, including core vaccines such as rabies, FHV-1, FCV, and FPV [Source 2].
- **Oral health:** lifelong proactive dental care should start with kitten visits [Source 3].
- **General hygiene and safety:** regular hand h

### 🏗️ Activity Notes


The query "What preventive care is recommended for cats?" was chosen.

- Setting changed: Changed chunk size to 1500

- Before: Used only sources 1 and 4, sources 2 and 3 were irrelevant. I suspected that since sources 2, and 3 had context too narrow to be useful, by increasing the chunk size we can get a better chance for a chunk not only to have similar textual meaning, but also hold more relevant context. 
- After: All 4 sources had relevant context to answer the query and were used by the generated answer. 
- Did retrieval improve? Why or why not? The retrieval improved as now all 4 chunks were used as sources, and had much more speficic answers to the query. Previously as suspected, the retrieved chunks were too short to have enough context to directly answer the query. 
